# Capstone modelling: predicting serious delinquency in consumer credit

Group 4 | October 2026 | eHA Academy Data Science Capstone, FinTech Risk and Fraud track

This notebook continues from 01_eda.ipynb. It reads the training and test parts saved there. The test part stays untouched until the final evaluation. Every value that changes the data, such as a filling value or a cap, is computed from the training part only.

# Module 6-7: Machine learning

## Load the training and test parts

In [9]:
# Import the libraries used in this notebook
import os
import numpy as np
import pandas as pd

# Show where this notebook is running and what the processed folder holds
print("Working folder:", os.getcwd())
print("Files in processed:", os.listdir("processed") if os.path.exists("processed") else "folder not found")

# Read the training and test parts saved by 01_eda.ipynb
train = pd.read_csv("processed/train.csv")
test = pd.read_csv("processed/test.csv")

# Show the size and delinquency rate of each part
print("Train:", train.shape, "| delinquency rate:", round(train["SeriousDlqin2yrs"].mean() * 100, 2), "%")
print("Test: ", test.shape, "| delinquency rate:", round(test["SeriousDlqin2yrs"].mean() * 100, 2), "%")

# Show the columns that still have missing values in the training part
print(train.isna().sum()[train.isna().sum() > 0])

Working folder: c:\Users\talk2\OneDrive\Documents\eHA\Data_Science\Capstone
Files in processed: ['test.csv', 'train.csv']
Train: (119512, 19) | delinquency rate: 6.7 %
Test:  (29878, 19) | delinquency rate: 6.7 %
NumberOfTime30-59DaysPastDueNotWorse      177
DebtRatio                               23351
MonthlyIncome                           23351
NumberOfTimes90DaysLate                   177
NumberOfTime60-89DaysPastDueNotWorse      177
NumberOfDependents                       3087
total_late                                177
late_group                              95218
dtype: int64


## Cleaning, stage B (training part only)

Every fill value and cap is computed from the training part and then applied unchanged to the test part, so nothing from the test part reaches training.

1. Drop the columns that are not model inputs: delinquent (a text copy of the target, which would leak the answer), the four band columns (they repeat the numeric columns) and total_late (a sum of three inputs already in the table).
2. Fill missing monthly income, debt ratio, dependents and the three past-due columns with the training median. The flags income_missing_flag and past_due_code_flag keep the fact that these values were missing.
3. Cap utilisation, debt ratio, monthly income, open credit lines, real estate loans and dependents at the training 99th percentile. Past-due counts are not capped.

In [10]:
# Check why late_group shows missing values: the text "None" is read as missing by pandas
print("late_group missing:", train["late_group"].isna().sum(), "| rows with total_late == 0:", (train["total_late"] == 0).sum())

# Columns that are not model inputs
drop_cols = ["delinquent", "age_group", "income_band", "utilisation_band", "late_group", "total_late"]

# Names of the three past-due columns
due_cols = ["NumberOfTime30-59DaysPastDueNotWorse", "NumberOfTimes90DaysLate", "NumberOfTime60-89DaysPastDueNotWorse"]

# Columns filled with the training median
fill_cols = ["MonthlyIncome", "DebtRatio", "NumberOfDependents"] + due_cols

# Columns capped at the training 99th percentile
cap_cols = ["RevolvingUtilizationOfUnsecuredLines", "DebtRatio", "MonthlyIncome",
            "NumberOfOpenCreditLinesAndLoans", "NumberRealEstateLoansOrLines", "NumberOfDependents"]

# Drop the non-input columns from both parts
tr = train.drop(columns=drop_cols)
te = test.drop(columns=drop_cols)

# Fill values and caps come from the training part only
medians = tr[fill_cols].median()
caps = tr[cap_cols].quantile(0.99)
print("Fill values from the training part:")
print(medians.round(2))
print("Caps from the training part:")
print(caps.round(2))

# Number of training values above each cap, before capping
print("Training values above the cap:")
print((tr[cap_cols] > caps).sum())

# Function that applies the stored fill values and caps to any part
def apply_stage_b(df):
    out = df.copy()
    out[fill_cols] = out[fill_cols].fillna(medians)
    out[cap_cols] = out[cap_cols].clip(upper=caps, axis=1)
    return out

# Apply the same values to the training and test parts
tr = apply_stage_b(tr)
te = apply_stage_b(te)

# Check that no missing values remain and that every column is numeric
print("Missing values left:", tr.isna().sum().sum(), "in train |", te.isna().sum().sum(), "in test")
print("Shapes:", tr.shape, te.shape)
print(tr.dtypes)

# Save the model-ready parts so the whole group uses the same files
tr.to_csv("processed/train_model.csv", index=False)
te.to_csv("processed/test_model.csv", index=False)
print("Saved train_model.csv and test_model.csv")

late_group missing: 95218 | rows with total_late == 0: 95218
Fill values from the training part:
MonthlyIncome                           5400.0
DebtRatio                                  0.3
NumberOfDependents                         0.0
NumberOfTime30-59DaysPastDueNotWorse       0.0
NumberOfTimes90DaysLate                    0.0
NumberOfTime60-89DaysPastDueNotWorse       0.0
dtype: float64
Caps from the training part:
RevolvingUtilizationOfUnsecuredLines        1.09
DebtRatio                                 640.40
MonthlyIncome                           25000.00
NumberOfOpenCreditLinesAndLoans            25.00
NumberRealEstateLoansOrLines                4.00
NumberOfDependents                          4.00
Name: 0.99, dtype: float64
Training values above the cap:
RevolvingUtilizationOfUnsecuredLines    1196
DebtRatio                                962
MonthlyIncome                            949
NumberOfOpenCreditLinesAndLoans          934
NumberRealEstateLoansOrLines            1168


In [11]:
# Confirm that no missing values remain and check the shapes
print("Missing values left:", tr.isna().sum().sum(), "in train |", te.isna().sum().sum(), "in test")
print("Shapes:", tr.shape, te.shape)

Missing values left: 0 in train | 0 in test
Shapes: (119512, 13) (29878, 13)


After stage B no missing values remain, 0 in the training part and 0 in the test part. Each part has 13 columns: the target, the 10 inputs and the 2 flags.

## Baseline: logistic regression

A scaled logistic regression with class weights for the 6.7% imbalance. It is scored on out-of-fold probabilities from 5 stratified folds of the training part, so every borrower is scored by a model that did not see them. The test part is not used. The target from the problem statement is recall of at least 60% with precision of at least 20%.

In [12]:
# Import the model, scaling and cross-validation tools
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import average_precision_score, precision_recall_curve, roc_auc_score

# Inputs and target from the model-ready training part
X_tr = tr.drop(columns=["SeriousDlqin2yrs"])
y_tr = tr["SeriousDlqin2yrs"]

# Baseline: scaled logistic regression with class weights for the imbalance
baseline = make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=1000))

# Out-of-fold probabilities from 5 stratified folds
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
proba = cross_val_predict(baseline, X_tr, y_tr, cv=cv, method="predict_proba")[:, 1]

# Headline scores, with the PR-AUC of random guessing for comparison
print("PR-AUC:", round(average_precision_score(y_tr, proba), 3), "| ROC-AUC:", round(roc_auc_score(y_tr, proba), 3))
print("PR-AUC of random guessing:", round(y_tr.mean(), 3))

# Precision and recall at every threshold
prec, rec, thr = precision_recall_curve(y_tr, proba)

# Highest threshold that still reaches 60% recall, and the precision at that threshold
best = np.where(rec[:-1] >= 0.60)[0].max()
print("Threshold for 60% recall:", round(thr[best], 3), "| precision there:", round(prec[best], 3))

PR-AUC: 0.38 | ROC-AUC: 0.855
PR-AUC of random guessing: 0.067
Threshold for 60% recall: 0.639 | precision there: 0.303


### What the baseline shows

The baseline logistic regression reaches a PR-AUC of 0.380 against 0.067 for random guessing, which is about 5.7 times higher, and a ROC-AUC of 0.855. At the threshold that catches 60% of delinquent borrowers (0.639), precision is 30.3%. Both targets in the problem statement, recall of at least 60% and precision of at least 20%, are met on the training part.

In practice, reaching 60% recall means flagging about 13% of borrowers (0.6 x 0.067 / 0.303), and about 3 in 10 of the flagged borrowers become delinquent, against 6.7% overall.

Two limits apply. The scores are out-of-fold on the training part, and the threshold was chosen from the same predictions, so precision is slightly optimistic. The final evaluation applies this threshold, fixed from the training part, to the test part. The model uses class weights, so 0.639 is a cut-off and not a probability of default.

In [13]:
# Import the stronger model
from sklearn.ensemble import HistGradientBoostingClassifier

# Gradient boosting with early stopping inside each fold
gb = HistGradientBoostingClassifier(learning_rate=0.05, max_iter=300, max_depth=4,
                                    early_stopping=True, validation_fraction=0.1,
                                    n_iter_no_change=20, random_state=42)

# Out-of-fold probabilities from the same 5 folds as the baseline
proba_gb = cross_val_predict(gb, X_tr, y_tr, cv=cv, method="predict_proba")[:, 1]

# Function that scores one model and returns one row for the experiment log
def score(name, p):
    prec, rec, thr = precision_recall_curve(y_tr, p)
    best = np.where(rec[:-1] >= 0.60)[0].max()
    return {"model": name,
            "PR-AUC": round(average_precision_score(y_tr, p), 3),
            "ROC-AUC": round(roc_auc_score(y_tr, p), 3),
            "threshold_at_60_recall": round(thr[best], 3),
            "precision_at_60_recall": round(prec[best], 3)}

# Experiment log with the baseline and the new model
log = pd.DataFrame([score("Logistic regression (baseline)", proba),
                    score("Gradient boosting", proba_gb)])
print(log.to_string(index=False))

# Save the log so the whole group has the same record
log.to_csv("processed/experiment_log.csv", index=False)

                         model  PR-AUC  ROC-AUC  threshold_at_60_recall  precision_at_60_recall
Logistic regression (baseline)   0.380    0.855                   0.639                   0.303
             Gradient boosting   0.396    0.865                   0.134                   0.326


### What the experiment log shows

Gradient boosting scores a PR-AUC of 0.396 against 0.380 for the baseline, and a ROC-AUC of 0.865 against 0.855. At 60% recall its precision is 32.6% against 30.3%. At that precision, the flagged group is about 12.3% of borrowers instead of 13.3%, which saves about 1 flagged borrower in 13 for the same recall. Both models meet the targets in the problem statement of recall of at least 60% and precision of at least 20%. The PR-AUC of random guessing is 0.067, so both models are about 6 times better than guessing.

The gain is small. These scores come from one run of 5 folds, so the spread between folds is not yet known. The thresholds are not comparable: the baseline uses class weights, so its threshold of 0.639 is a cut-off, while gradient boosting has no class weights, so its threshold of 0.134 should be close to a predicted probability of delinquency.

## Is the gain real?

The scores of both models are compared fold by fold. If gradient boosting wins in every fold, the gain is consistent. If the results overlap, the gain could be chance.

In [14]:
# PR-AUC of each model in each of the 5 folds, using the same folds as before
from sklearn.model_selection import cross_val_score

# Score both models fold by fold
base_folds = cross_val_score(baseline, X_tr, y_tr, cv=cv, scoring="average_precision")
gb_folds = cross_val_score(gb, X_tr, y_tr, cv=cv, scoring="average_precision")

# Table of the fold scores and the difference per fold
folds = pd.DataFrame({"fold": range(1, 6), "baseline": base_folds.round(3),
                      "gradient_boosting": gb_folds.round(3),
                      "difference": (gb_folds - base_folds).round(3)})
print(folds.to_string(index=False))

# Average and spread of each model across the folds
print("Baseline: mean", round(base_folds.mean(), 3), "| std", round(base_folds.std(), 3))
print("Gradient boosting: mean", round(gb_folds.mean(), 3), "| std", round(gb_folds.std(), 3))

 fold  baseline  gradient_boosting  difference
    1     0.380              0.397       0.017
    2     0.385              0.403       0.018
    3     0.378              0.384       0.006
    4     0.376              0.393       0.017
    5     0.386              0.406       0.021
Baseline: mean 0.381 | std 0.004
Gradient boosting: mean 0.397 | std 0.008


### Is the gain real: verdict

Gradient boosting scores a higher PR-AUC than the baseline in all 5 folds, by 0.006 to 0.021. The mean is 0.397 against 0.381, a difference of about 0.016. The baseline varies little between folds (std 0.004) and gradient boosting varies a bit more (0.008). The smallest gain, 0.006 in fold 3, is small, so the gain is consistent but modest. The folds share training rows, so this is not a formal significance test.

Decision: gradient boosting is the final model, because it is better in every fold. The logistic regression stays in the report as the comparison point. Its coefficients can be read one by one, which matters if the lender has to explain a decision to a borrower.

## What the model relies on

Permutation importance on rows the model did not train on. The training part is split again 80/20, the model is fitted on the 80%, and each input is shuffled in turn on the 20%. The drop in PR-AUC shows how much the model needs that input. The test part is not used.

In [15]:
# Import tools to copy the model and measure permutation importance
from sklearn.base import clone
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split

# Split the training part again, so importance is measured on rows the model did not train on
X_fit, X_val, y_fit, y_val = train_test_split(X_tr, y_tr, test_size=0.2, stratify=y_tr, random_state=42)

# Fit a fresh copy of gradient boosting on the 80% part
gb_fit = clone(gb).fit(X_fit, y_fit)

# Shuffle each input in turn on the 20% part and record the drop in PR-AUC
imp = permutation_importance(gb_fit, X_val, y_val, scoring="average_precision", n_repeats=5, random_state=42)

# Table of the drops, largest first
imp_table = pd.DataFrame({"input": X_tr.columns,
                          "drop_in_PR_AUC": imp.importances_mean.round(4),
                          "std": imp.importances_std.round(4)}).sort_values("drop_in_PR_AUC", ascending=False)
print(imp_table.to_string(index=False))

                               input  drop_in_PR_AUC    std
             NumberOfTimes90DaysLate          0.1087 0.0041
RevolvingUtilizationOfUnsecuredLines          0.0778 0.0046
NumberOfTime30-59DaysPastDueNotWorse          0.0402 0.0017
NumberOfTime60-89DaysPastDueNotWorse          0.0223 0.0012
     NumberOfOpenCreditLinesAndLoans          0.0159 0.0016
                           DebtRatio          0.0143 0.0029
                                 age          0.0118 0.0021
                  past_due_code_flag          0.0086 0.0003
                       MonthlyIncome          0.0065 0.0018
        NumberRealEstateLoansOrLines          0.0051 0.0016
                  NumberOfDependents          0.0033 0.0009
                 income_missing_flag         -0.0002 0.0003


## Where the model fails

The out-of-fold predictions of gradient boosting at the threshold that gives 60% recall. The confusion counts show the cost of the choice, and the recall by group shows which delinquent borrowers the model misses.

In [16]:
# Threshold for 60% recall, read from the experiment log
t = log.loc[log["model"] == "Gradient boosting", "threshold_at_60_recall"].iloc[0]

# Borrowers the model flags at that threshold
flag = pd.Series(proba_gb >= t, index=y_tr.index)

# Confusion counts
tp = int((flag & (y_tr == 1)).sum())
fp = int((flag & (y_tr == 0)).sum())
fn = int((~flag & (y_tr == 1)).sum())
tn = int((~flag & (y_tr == 0)).sum())
print(f"Threshold {t} | flagged {tp + fp} | caught {tp} | false alarms {fp} | missed {fn} | not flagged and fine {tn}")
print(f"Recall {tp / (tp + fn):.3f} | precision {tp / (tp + fp):.3f}")

# Groups for the error analysis, taken from columns kept in the train table
late_any = tr[due_cols].sum(axis=1) > 0
groups = pd.DataFrame({"age_group": train["age_group"],
                       "late_payments": np.where(late_any, "Some late payments", "No late payments"),
                       "y": y_tr, "flag": flag})

# Share of delinquent borrowers the model catches in each group
for col in ["age_group", "late_payments"]:
    table = groups[groups["y"] == 1].groupby(col)["flag"].agg(delinquent="count", caught="mean")
    table["caught"] = (table["caught"] * 100).round(1)
    print(table)
    print()

Threshold 0.134 | flagged 14730 | caught 4807 | false alarms 9923 | missed 3200 | not flagged and fine 101582
Recall 0.600 | precision 0.326
           delinquent  caught
age_group                    
18-29             814    64.3
30-44            2982    65.4
45-59            3021    59.3
60+              1190    45.6

                    delinquent  caught
late_payments                         
No late payments          2721    13.1
Some late payments        5286    84.2



### What the model relies on: result

The past-due columns and utilisation carry the model. Shuffling NumberOfTimes90DaysLate lowers the PR-AUC the most (0.1087), followed by utilisation (0.0778), the 30 to 59 days column (0.0402) and the 60 to 89 days column (0.0223). Open credit lines (0.0159), debt ratio (0.0143), age (0.0118), past_due_code_flag (0.0086), monthly income (0.0065), real estate loans (0.0051) and dependents (0.0033) follow. income_missing_flag is at -0.0002 with a std of 0.0003, so it adds nothing.

This matches the EDA, where late payments and utilisation separated the two groups most clearly. The drops rank the inputs and do not split the score into parts, because inputs that overlap in information share their importance. past_due_code_flag adds 0.0086 even though the past-due values in those rows were filled with 0, so the flag carries information that the filled values do not.

### Where the model fails: result

At the threshold of 0.134 the model flags 14,730 of the 119,512 training borrowers (12.3%). It catches 4,807 of the 8,007 delinquent borrowers (recall 0.600) and misses 3,200. The flagged group holds 9,923 false alarms, about 2 for every delinquent borrower caught, and precision is 0.326.

The misses are concentrated. Among delinquent borrowers with some late payments, the model catches 84.2% (5,286 borrowers). Among delinquent borrowers with no late payments, it catches 13.1% (2,721 borrowers, about a third of all delinquent borrowers). That group makes up about three quarters of the 3,200 misses. The model finds borrowers who are already paying late and mostly fails to find those who default with a clean payment history. The no late payments group includes the 177 flagged code rows, whose past-due values were filled with 0.

By age, recall is 64.3% for 18 to 29, 65.4% for 30 to 44, 59.3% for 45 to 59 and 45.6% for 60 and over. The model misses more of the oldest borrowers. We did not test why.

A lender using this score at 60% recall still misses 40% of its delinquent borrowers, and about three quarters of those had no late payments. The report states this as a limit.

## Which inputs are needed

Gradient boosting is refitted with one input left out at a time and scored with the same 5 folds. The first row repeats the full model as a check and should match the experiment log. If a row scores about the same as the full model, that input is not needed for the score.

In [17]:
# Inputs to leave out, one test at a time
tests = {"All inputs": [],
         "Without past_due_code_flag": ["past_due_code_flag"],
         "Without age": ["age"],
         "Without income_missing_flag": ["income_missing_flag"]}

# Score gradient boosting for each version of the inputs
rows = []
for name, drop in tests.items():
    # Keep every input except the ones left out
    cols = [c for c in X_tr.columns if c not in drop]
    # Out-of-fold probabilities with the same folds as before
    p = cross_val_predict(gb, X_tr[cols], y_tr, cv=cv, method="predict_proba")[:, 1]
    # One row for the table, using the score function from the experiment log
    rows.append(score(name, p))

# Print the table and add the rows to the experiment log file
ablation = pd.DataFrame(rows)
print(ablation.to_string(index=False))
ablation.to_csv("processed/ablation.csv", index=False)

                      model  PR-AUC  ROC-AUC  threshold_at_60_recall  precision_at_60_recall
                 All inputs   0.396    0.865                   0.134                   0.326
 Without past_due_code_flag   0.391    0.864                   0.137                   0.328
                Without age   0.393    0.862                   0.130                   0.322
Without income_missing_flag   0.396    0.864                   0.134                   0.326


### Which inputs are needed: result

Leaving out one input at a time changes the PR-AUC by 0.005 or less. Without past_due_code_flag it is 0.391 (0.005 lower), without age 0.393 (0.003 lower), and without income_missing_flag 0.396 (no change). Each change is smaller than the fold-to-fold spread of gradient boosting (std 0.008), and the table comes from one run, so the differences are not conclusive. At 60% recall, precision is 0.328 without the code flag, 0.322 without age and 0.326 without the income flag, against 0.326 for the full model.

Age shows a drop of 0.0118 in the importance table but costs only 0.003 when removed. The reason is that other inputs, such as income and open credit lines, overlap with age, so the model recovers most of its information from them.

Decisions: income_missing_flag is dropped because it adds nothing. past_due_code_flag is dropped because it may reflect how the competition data was built, a real lender may not have such codes, and the model loses only 0.005 PR-AUC and no precision without it. Age is dropped because it is a sensitive input in lending and the loss is 0.003 PR-AUC and 0.004 precision. These are my proposals, and the group should confirm them. The three removals were tested one at a time, so the combined set is tested next.

## Final inputs

The final model leaves out income_missing_flag, past_due_code_flag and age. The rule was set before the run: the smaller input set is accepted if its PR-AUC stays within 0.01 of the full model (0.396), which is about the fold-to-fold spread. The recall by age group is checked again, because age is no longer an input.

In [18]:
# Inputs left out of the final model
final_drop = ["income_missing_flag", "past_due_code_flag", "age"]

# Keep every other input
final_cols = [c for c in X_tr.columns if c not in final_drop]
print("Final inputs:", final_cols)

# Out-of-fold probabilities for the final input set, with the same folds as before
proba_final = cross_val_predict(gb, X_tr[final_cols], y_tr, cv=cv, method="predict_proba")[:, 1]

# Score it like the other models and add it to the experiment log
row = score("Gradient boosting, final inputs", proba_final)
print(row)
log = pd.concat([log, pd.DataFrame([row])], ignore_index=True)
log.to_csv("processed/experiment_log.csv", index=False)

# Flag the borrowers above the 60% recall threshold of the final model
flag_final = pd.Series(proba_final >= row["threshold_at_60_recall"], index=y_tr.index)

# Share of delinquent borrowers caught in each age group
table = pd.DataFrame({"age_group": train["age_group"], "y": y_tr, "flag": flag_final})
table = table[table["y"] == 1].groupby("age_group")["flag"].agg(delinquent="count", caught="mean")
table["caught"] = (table["caught"] * 100).round(1)
print(table)

Final inputs: ['RevolvingUtilizationOfUnsecuredLines', 'NumberOfTime30-59DaysPastDueNotWorse', 'DebtRatio', 'MonthlyIncome', 'NumberOfOpenCreditLinesAndLoans', 'NumberOfTimes90DaysLate', 'NumberRealEstateLoansOrLines', 'NumberOfTime60-89DaysPastDueNotWorse', 'NumberOfDependents']
{'model': 'Gradient boosting, final inputs', 'PR-AUC': 0.389, 'ROC-AUC': 0.861, 'threshold_at_60_recall': np.float64(0.134), 'precision_at_60_recall': np.float64(0.323)}
           delinquent  caught
age_group                    
18-29             814    62.0
30-44            2982    63.1
45-59            3021    60.2
60+              1190    50.7


### Final inputs: result

The final model uses 9 inputs: utilisation, the three past-due counts, debt ratio, monthly income, open credit lines, real estate loans and dependents. Its PR-AUC is 0.389 against 0.396 for the full model, a drop of 0.007, which is inside the 0.01 limit set before the run. ROC-AUC is 0.861 against 0.865, and precision at 60% recall is 0.323 against 0.326. The final input set is accepted.

Recall by age group, with the previous full-model figures in brackets: 62.0% for 18 to 29 (64.3%), 63.1% for 30 to 44 (65.4%), 60.2% for 45 to 59 (59.3%) and 50.7% for 60 and over (45.6%). Without age as an input, the oldest group is caught more often, and the two youngest groups slightly less. The oldest group is still the lowest, 9.5 points below the 45 to 59 group. A likely reason is that without age the model can no longer lower the scores of older borrowers, who default less, but this was not tested. Age is kept in the table only to check recall by group.

## Final evaluation on the test part

The final model is fitted on the whole training part with the final inputs. The threshold of 0.134 was fixed earlier from the out-of-fold predictions on the training part. The test part is used once, with no changes made afterwards. Confidence intervals come from 1,000 bootstrap resamples of the test rows. The recall target in the problem statement is 60% and the precision target is 20%.

In [19]:
# Fit the final model on the whole training part, using only the final inputs
gb_final = clone(gb).fit(X_tr[final_cols], y_tr)

# Test inputs and target, with the same columns
X_te = te[final_cols]
y_te = te["SeriousDlqin2yrs"].values

# Predicted probability of delinquency for each test borrower
p_te = gb_final.predict_proba(X_te)[:, 1]

# Threshold fixed from the training part before this step
t_final = row["threshold_at_60_recall"]

# Borrowers flagged at that threshold
flag_te = p_te >= t_final

# Confusion counts on the test part
tp = int((flag_te & (y_te == 1)).sum())
fp = int((flag_te & (y_te == 0)).sum())
fn = int((~flag_te & (y_te == 1)).sum())
tn = int((~flag_te & (y_te == 0)).sum())
print(f"Flagged {tp + fp} of {len(y_te)} | caught {tp} | false alarms {fp} | missed {fn} | fine and not flagged {tn}")
print(f"Recall {tp / (tp + fn):.3f} | precision {tp / (tp + fp):.3f}")
print(f"PR-AUC {average_precision_score(y_te, p_te):.3f} | ROC-AUC {roc_auc_score(y_te, p_te):.3f} | random PR-AUC {y_te.mean():.3f}")

# Bootstrap 95% intervals for recall and precision
rng = np.random.default_rng(42)
rec_b, prec_b = [], []
for _ in range(1000):
    idx = rng.integers(0, len(y_te), len(y_te))
    f, y_b = flag_te[idx], y_te[idx]
    rec_b.append((f & (y_b == 1)).sum() / (y_b == 1).sum())
    prec_b.append((f & (y_b == 1)).sum() / f.sum())
print("Recall 95% interval:", np.percentile(rec_b, [2.5, 97.5]).round(3))
print("Precision 95% interval:", np.percentile(prec_b, [2.5, 97.5]).round(3))

# Recall by age group on the test part, using the age_group column kept in the test file
by_age = pd.DataFrame({"age_group": test["age_group"].values, "y": y_te, "flag": flag_te})
by_age = by_age[by_age["y"] == 1].groupby("age_group")["flag"].agg(delinquent="count", caught="mean")
by_age["caught"] = (by_age["caught"] * 100).round(1)
print(by_age)

Flagged 3846 of 29878 | caught 1240 | false alarms 2606 | missed 762 | fine and not flagged 25270
Recall 0.619 | precision 0.322
PR-AUC 0.406 | ROC-AUC 0.861 | random PR-AUC 0.067
Recall 95% interval: [0.598 0.641]
Precision 95% interval: [0.309 0.338]
           delinquent  caught
age_group                    
18-29             212    69.3
30-44             714    65.4
45-59             766    60.1
60+               310    53.5


### Final evaluation: result

The final model flags 3,846 of the 29,878 test borrowers (12.9%). It catches 1,240 of the 2,002 delinquent borrowers (recall 0.619) and misses 762. The flagged group holds 2,606 false alarms, and precision is 0.322. The bootstrap 95% intervals are 0.598 to 0.641 for recall and 0.309 to 0.338 for precision. PR-AUC is 0.406 and ROC-AUC is 0.861, against a random PR-AUC of 0.067.

Both targets in the problem statement are met on the test part, with one caveat. The recall of 0.619 is above 60%, but the lower end of its interval, 0.598, is just below it, so the data do not show that the recall target is met with certainty. Precision is well above the 20% target: about 32 of every 100 flagged borrowers become delinquent, against 6.7% overall.

The test PR-AUC of 0.406 is higher than the out-of-fold figure of 0.389 on the training part. The test part holds 2,002 delinquent borrowers, and a difference of 0.017 can come from the sample. The test part was used once and nothing was changed after seeing the result.

Recall by age group is 69.3% for 18 to 29, 65.4% for 30 to 44, 60.1% for 45 to 59 and 53.5% for 60 and over. Age is not a model input, yet recall still falls with age, and the oldest group is 6.6 points below the 45 to 59 group. We did not test why. The 18 to 29 group has only 212 delinquent borrowers, so its figure is uncertain.

Limits: the test part is a random 20% of the same file, so it says nothing about borrowers at a later date. The file gives no country or year, and the model was not tested out of time or in another market.

In [20]:
# Import pickle to write the model bundle to disk
import pickle

# Bundle everything the demo app needs to score a new borrower the same way
bundle = {
    # The fitted gradient boosting model
    "model": gb_final,
    # Input columns in the order the model expects
    "inputs": final_cols,
    # Threshold fixed from the training part
    "threshold": float(t_final),
    # Training medians used to fill missing values
    "medians": medians.to_dict(),
    # Training 99th percentile caps
    "caps": caps.to_dict(),
}

# Write the bundle to the processed folder
with open("processed/model.pkl", "wb") as f:
    pickle.dump(bundle, f)

# Confirm what was saved
print("Saved processed/model.pkl with inputs:", final_cols)

Saved processed/model.pkl with inputs: ['RevolvingUtilizationOfUnsecuredLines', 'NumberOfTime30-59DaysPastDueNotWorse', 'DebtRatio', 'MonthlyIncome', 'NumberOfOpenCreditLinesAndLoans', 'NumberOfTimes90DaysLate', 'NumberRealEstateLoansOrLines', 'NumberOfTime60-89DaysPastDueNotWorse', 'NumberOfDependents']


In [21]:
# Import scikit-learn and print the installed version
import sklearn
print(sklearn.__version__)

1.9.1


In [22]:
# Import the libraries the saved model depends on
import sklearn, numpy, pandas, scipy

# Print their versions so the app can pin the same ones
print("scikit-learn", sklearn.__version__)
print("numpy", numpy.__version__)
print("pandas", pandas.__version__)
print("scipy", scipy.__version__)

scikit-learn 1.9.1
numpy 2.5.4
pandas 3.0.6
scipy 1.18.1
